In [1]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma

import gradio as gr

load_dotenv()

# ── Data ──────────────────────────────────────────────────────────────────────
books = pd.read_csv("books_with_emotions.csv")
books["large_thumbnail"] = books["thumbnail"] + "&fife=w800"
books["large_thumbnail"] = np.where(
    books["large_thumbnail"].isna(), "cover-not-found.jpg", books["large_thumbnail"]
)

raw_documents = TextLoader("tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=0)
documents     = text_splitter.split_documents(raw_documents)
db_books      = Chroma.from_documents(
    documents, HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

# ── Logic ──────────────────────────────────────────────────────────────────────
TONE_COL = {
    "Happy": "joy", "Surprising": "surprise",
    "Angry": "anger", "Suspenseful": "fear", "Sad": "sadness",
}

def retrieve_semantic_recommendations(
    query: str,
    category: str = None,
    tone: str = None,
    initial_top_k: int = 50,
    final_top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k=initial_top_k)
    books_list = [int(r.page_content.strip('"').split()[0]) for r in recs]
    book_recs  = books[books["isbn13"].isin(books_list)].head(initial_top_k)

    if category and category != "All":
        book_recs = book_recs[book_recs["simple_categories"] == category]
    book_recs = book_recs.head(final_top_k)

    if tone and tone in TONE_COL:
        book_recs = book_recs.sort_values(by=TONE_COL[tone], ascending=False)

    return book_recs


def recommend_books(query: str, category: str, tone: str):
    recs    = retrieve_semantic_recommendations(query, category, tone)
    results = []
    for _, row in recs.iterrows():
        desc   = " ".join(row["description"].split()[:30]) + "..."
        parts  = row["authors"].split(";")
        if len(parts) == 2:
            author = f"{parts[0]} and {parts[1]}"
        elif len(parts) > 2:
            author = f"{', '.join(parts[:-1])}, and {parts[-1]}"
        else:
            author = row["authors"]
        caption = f"{row['title']} — {author}\n{desc}"
        results.append((row["large_thumbnail"], caption))
    return results


categories = ["All"] + sorted(books["simple_categories"].dropna().unique())
tones      = ["All", "Happy", "Surprising", "Angry", "Suspenseful", "Sad"]
n_books    = f"{len(books):,}"

# ── CSS ────────────────────────────────────────────────────────────────────────
css = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:ital,wght@0,400;0,700;1,400;1,600&family=Outfit:wght@300;400;500;600&display=swap');

*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

:root {
  --ink:       #0c0e12;
  --paper:     #f5f0e8;
  --paper2:    #ece7dc;
  --paper3:    #e0d9cc;
  --gold:      #a07840;
  --gold-l:    #c8a060;
  --gold-pale: #f0e4cc;
  --rust:      #8b3a2a;
  --text:      #1a1610;
  --muted:     #7a6e5e;
  --border:    #d4c9b4;
}

body, .gradio-container {
  background: var(--paper) !important;
  font-family: 'Outfit', sans-serif !important;
  color: var(--text) !important;
}

.gradio-container { max-width: 1160px !important; margin: 0 auto !important; }

/* hide default gradio chrome */
footer, .built-with { display: none !important; }

/* ── Header ── */
#bibliophile-header {
  display: grid;
  grid-template-columns: 1fr auto;
  align-items: end;
  gap: 24px;
  padding: 52px 0 30px;
  border-bottom: 1.5px solid var(--border);
  margin-bottom: 36px;
}
#bibliophile-header .eyebrow {
  font-size: 0.63rem; font-weight: 600; letter-spacing: 0.2em;
  text-transform: uppercase; color: var(--gold);
  display: flex; align-items: center; gap: 10px; margin-bottom: 14px;
}
#bibliophile-header .eyebrow::before {
  content: ''; width: 24px; height: 1px; background: var(--gold);
}
#bibliophile-header h1 {
  font-family: 'Playfair Display', serif !important;
  font-size: 2.9rem !important; font-weight: 400 !important;
  line-height: 1.08 !important; letter-spacing: -0.02em !important;
  color: var(--ink) !important;
}
#bibliophile-header h1 em { font-style: italic; color: var(--gold); }
.header-stats { text-align: right; }
.stat-block { margin-bottom: 12px; }
.stat-block .num {
  font-family: 'Playfair Display', serif;
  font-size: 2rem; font-weight: 700; color: var(--ink); line-height: 1;
}
.stat-block .lbl {
  font-size: 0.68rem; color: var(--muted); letter-spacing: 0.05em; margin-top: 2px;
}

/* ── Search bar ── */
#search-wrap { margin-bottom: 14px; }
#search-wrap label {
  font-size: 0.63rem !important; font-weight: 600 !important;
  letter-spacing: 0.15em !important; text-transform: uppercase !important;
  color: var(--muted) !important; margin-bottom: 10px !important;
}
#query-input textarea {
  background: #fff !important;
  border: 1.5px solid var(--border) !important;
  border-radius: 12px !important;
  color: var(--text) !important;
  font-family: 'Outfit', sans-serif !important;
  font-size: 1rem !important; font-weight: 300 !important;
  padding: 18px 22px !important;
  resize: none !important;
  box-shadow: 0 2px 12px rgba(0,0,0,0.06) !important;
  transition: border-color 0.2s, box-shadow 0.2s !important;
  min-height: 56px !important;
}
#query-input textarea:focus {
  border-color: var(--gold) !important;
  box-shadow: 0 4px 20px rgba(160,120,64,0.15) !important;
  outline: none !important;
}
#query-input textarea::placeholder { color: #b5aa9a !important; }

/* ── Filter selects ── */
#cat-select label, #tone-select label {
  font-size: 0.63rem !important; font-weight: 600 !important;
  letter-spacing: 0.15em !important; text-transform: uppercase !important;
  color: var(--muted) !important;
}
#cat-select select, #tone-select select {
  background: #fff !important;
  border: 1.5px solid var(--border) !important;
  border-radius: 10px !important;
  color: var(--text) !important;
  font-family: 'Outfit', sans-serif !important;
  font-size: 0.9rem !important; font-weight: 400 !important;
  padding: 12px 16px !important;
  box-shadow: 0 2px 8px rgba(0,0,0,0.05) !important;
  transition: border-color 0.2s !important;
}
#cat-select select:focus, #tone-select select:focus {
  border-color: var(--gold) !important; outline: none !important;
}

/* ── Submit button ── */
#submit-btn button {
  width: 100% !important;
  background: var(--ink) !important;
  color: var(--paper) !important;
  border: none !important;
  border-radius: 10px !important;
  font-family: 'Outfit', sans-serif !important;
  font-size: 0.78rem !important; font-weight: 600 !important;
  letter-spacing: 0.12em !important; text-transform: uppercase !important;
  padding: 15px 20px !important; height: 52px !important;
  cursor: pointer !important;
  transition: background 0.2s, transform 0.1s !important;
}
#submit-btn button:hover  { background: #2a2620 !important; }
#submit-btn button:active { transform: scale(0.97) !important; }

/* ── Results section ── */
#results-header {
  display: flex; align-items: baseline;
  justify-content: space-between;
  padding-bottom: 14px;
  border-bottom: 1px solid var(--border);
  margin-bottom: 22px;
}
#results-header .rt {
  font-family: 'Playfair Display', serif;
  font-size: 1.15rem; color: var(--ink); font-style: italic;
}
#results-header .rm {
  font-size: 0.7rem; color: var(--muted);
}

/* ── Gallery ── */
#gallery-out { background: transparent !important; border: none !important; }
#gallery-out .grid-wrap { gap: 14px !important; }

#gallery-out .gallery-item {
  border-radius: 8px !important;
  overflow: hidden !important;
  border: 1px solid var(--border) !important;
  background: #fff !important;
  transition: transform 0.2s, box-shadow 0.2s !important;
  box-shadow: 0 3px 10px rgba(0,0,0,0.08) !important;
}
#gallery-out .gallery-item:hover {
  transform: translateY(-6px) !important;
  box-shadow: 0 16px 36px rgba(0,0,0,0.16) !important;
  border-color: var(--gold-pale) !important;
}
#gallery-out img {
  object-fit: cover !important;
  aspect-ratio: 2/3 !important;
  width: 100% !important;
}
#gallery-out .caption-label {
  font-family: 'Outfit', sans-serif !important;
  font-size: 0.7rem !important; line-height: 1.4 !important;
  color: var(--muted) !important;
  padding: 8px 10px !important;
  background: #fff !important;
}

/* ── Footer ── */
#bibliophile-footer {
  display: flex; justify-content: space-between; align-items: center;
  margin-top: 52px; padding-top: 18px; border-top: 1px solid var(--border);
}
#bibliophile-footer .brand {
  font-family: 'Playfair Display', serif;
  font-style: italic; font-size: 1rem; color: var(--muted);
}
#bibliophile-footer .tech {
  font-size: 0.62rem; letter-spacing: 0.1em;
  text-transform: uppercase; color: var(--border);
}

/* scrollbar */
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-track { background: var(--paper2); }
::-webkit-scrollbar-thumb { background: var(--border); border-radius: 3px; }
"""

# ── UI ─────────────────────────────────────────────────────────────────────────
with gr.Blocks(css=css, title="Bibliophile") as dashboard:

    gr.HTML(f"""
    <div id="bibliophile-header">
      <div>
        <div class="eyebrow">AI-powered discovery</div>
        <h1>Your next<br><em>favourite book</em><br>is one search away</h1>
      </div>
      <div class="header-stats">
        <div class="stat-block">
          <div class="num">{n_books}</div>
          <div class="lbl">cuốn sách</div>
        </div>
        <div class="stat-block">
          <div class="num">7</div>
          <div class="lbl">chiều cảm xúc</div>
        </div>
      </div>
    </div>
    """)

    with gr.Row():
        user_query = gr.Textbox(
            label="Mô tả cuốn sách bạn muốn đọc",
            placeholder="e.g., a haunting story about memory and loss set in wartime…",
            lines=1,
            elem_id="query-input",
            scale=5,
        )
        category_dropdown = gr.Dropdown(
            choices=categories, label="Thể loại", value="All",
            elem_id="cat-select", scale=1,
        )
        tone_dropdown = gr.Dropdown(
            choices=tones, label="Cảm xúc", value="All",
            elem_id="tone-select", scale=1,
        )
        submit_button = gr.Button(
            "Tìm kiếm →", variant="primary",
            elem_id="submit-btn", scale=1,
        )

    gr.HTML("""
    <div id="results-header">
      <div class="rt">Kết quả gợi ý</div>
      <div class="rm">16 cuốn phù hợp nhất</div>
    </div>
    """)

    output = gr.Gallery(
        label="", columns=8, rows=2,
        object_fit="cover", height="auto",
        elem_id="gallery-out", show_label=False,
    )

    gr.HTML("""
    <div id="bibliophile-footer">
      <div class="brand">Bibliophile</div>
      <div class="tech">Semantic search · Emotion analysis · Hybrid recommendation</div>
    </div>
    """)

    submit_button.click(
        fn=recommend_books,
        inputs=[user_query, category_dropdown, tone_dropdown],
        outputs=output,
    )
    user_query.submit(
        fn=recommend_books,
        inputs=[user_query, category_dropdown, tone_dropdown],
        outputs=output,
    )

if __name__ == "__main__":
    dashboard.launch()

C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Created a chunk of size 1168, which is longer than the specified 1000
Created a chunk of size 1214, which is longer than the specified 1000
Created a chunk of size 1088, which is longer than the specified 1000
Created a chunk of size 1189, which is longer than the specified 1000
Created a chunk of size 1267, which is longer than the specified 1000
Created a chunk of size 2010, which is longer than the specified 1000
Created a chunk of size 1225, which is longer than the specified 1000
Created a chunk of size 1184, which is longer than the specified 1000
Created a chunk of size 1214, which is longer than the specified 1000
Created a chunk of size 1191, which is longer than the specified 1000
Created a chunk of size 1057, which is longer than the specified 1000
Created a chunk of size 1270, which is longer than the specified 1000
Created a chunk of size 1635, which is longer than the specified 1000
Created a chunk of size 1132, which is longer than the specified 1000
Created a chunk of s

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import json
import re
import os
import zipfile
import numpy as np
from collections import Counter
from difflib import SequenceMatcher
from rank_bm25 import BM25Okapi

# ==========================================
# 1. TIỀN XỬ LÝ ĐA LĨNH VỰC
# ==========================================
def preprocess_text(text):
    """
    Tiền xử lý giữ nguyên vẹn cấu trúc câu, loại bỏ dấu câu thừa.
    Không xóa stopword để giữ nguyên ngữ nghĩa.
    """
    if not text:
        return ""
    text = text.lower()

    # Chuẩn hóa phân số, tỷ lệ (VD: 1 / 2 -> 1/2, 50 % -> 50%)
    text = re.sub(r'(\d+)[\s]*[/-][\s]*(\d+)', r'\1/\2', text)
    text = re.sub(r'(\d+)[\s]*%', r'\1%', text)

    # Xóa ký tự đặc biệt, chỉ giữ chữ, số, gạch chéo và phần trăm
    text = re.sub(r'[^\w\s/%]', ' ', text)
    text = ' '.join(text.split())
    return text

def extract_universal_entities(text, original_text=None):
    """
    Trích xuất các thực thể quan trọng áp dụng cho MỌI LĨNH VỰC
    """
    entities = []

    # 1. Bắt mọi con số (Năm, Khối lượng, Tỷ lệ...)
    numbers = re.findall(r'\b\d+(?:[.,]\d+)?\b', text)
    entities.extend([f"NUM_{n}" for n in numbers])

    # 2. Bắt các từ viết tắt IN HOA (Nếu có truyền original_text vào)
    if original_text:
        acronyms = re.findall(r'\b[A-Z]{2,}\b', original_text)
        entities.extend([f"ACR_{a}" for a in acronyms])

    # 3. Bắt các "Bẫy trắc nghiệm tuyệt đối"
    absolute_terms = [
        'tất cả', 'mọi', 'duy nhất', 'chỉ có', 'luôn luôn', 'chắc chắn',
        'không bao giờ', 'hoàn toàn', 'chủ yếu', 'trọng tâm'
    ]
    for term in absolute_terms:
        if term in text:
            entities.append(f"ABS_{term.replace(' ', '_')}")

    return entities

def calculate_semantic_similarity(text1, text2, orig1=None, orig2=None):
    """Tính độ tương đồng ngữ nghĩa bách khoa"""
    if not text1 or not text2:
        return 0

    text1_proc = preprocess_text(text1)
    text2_proc = preprocess_text(text2)

    # 1. Similarity dựa trên cụm từ liền nhau (Rất tốt cho định nghĩa Khoa học)
    seq_sim = SequenceMatcher(None, text1_proc, text2_proc).ratio()

    # 2. Similarity dựa trên Thực thể (Số, Viết tắt)
    entities1 = set(extract_universal_entities(text1_proc, orig1))
    entities2 = set(extract_universal_entities(text2_proc, orig2))
    entity_sim = len(entities1 & entities2) / max(len(entities1), len(entities2)) if entities1 and entities2 else 0

    # 3. Mức độ bao phủ từ vựng chung
    words1 = set(text1_proc.split())
    words2 = set(text2_proc.split())
    word_sim = len(words1 & words2) / max(len(words1), len(words2)) if words1 and words2 else 0

    # Cân bằng lại trọng số cho tài liệu chung
    return 0.5 * word_sim + 0.3 * seq_sim + 0.2 * entity_sim

# ==========================================
# 2. TÌM KIẾM (CHUNKING & BM25)
# ==========================================
def chunk_text(text, max_words=120, overlap=30):
    """
    Tăng kích thước chunk lên 120, overlap 30 vì văn bản đa lĩnh vực
    thường có các đoạn giải thích định nghĩa khá dài.
    """
    words = text.split()
    chunks = []
    if len(words) <= max_words:
        return [text]

    for i in range(0, len(words), max_words - overlap):
        chunk = " ".join(words[i:i + max_words])
        chunks.append(chunk)
    return chunks

def load_corpus_chunked(corpus_file="dataset.json"):
    chunks = []
    chunk_metadata = []
    try:
        with open(corpus_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    doc = json.loads(line)
                    title = doc.get('title', '')
                    content = doc.get('content', '')
                    text = f"{title} {content}"

                    doc_chunks = chunk_text(text, max_words=120, overlap=30)
                    for c in doc_chunks:
                        chunks.append(c)
                        chunk_metadata.append({'title': title, 'content': content})
                except json.JSONDecodeError:
                    continue
    except FileNotFoundError:
        print(f"Loi: Khong tim thay file {corpus_file}")

    return chunks, chunk_metadata

def find_best_document_bm25(question, chunks, chunk_metadata, bm25_model):
    tokenized_query = preprocess_text(question).split()
    scores = bm25_model.get_scores(tokenized_query)

    top_k = min(3, len(scores))
    top_indices = np.argsort(scores)[-top_k:][::-1]

    combined_text = ""
    combined_metadata = []
    for idx in top_indices:
        combined_text += " " + chunks[idx]
        combined_metadata.append(chunk_metadata[idx])
    return combined_text, combined_metadata

# ==========================================
# 3. GIÁM KHẢO CHẤM ĐIỂM (HEURISTICS)
# ==========================================
def check_general_negation(choice_text, doc_text):
    """Bắt bẫy phủ định chung cho mọi môn học"""
    negation_words = ['không', 'chưa', 'phi', 'ngoại trừ', 'trái với']

    choice_tokens = set(preprocess_text(choice_text).split())
    doc_tokens = set(preprocess_text(doc_text).split())

    # Kiểm tra xem có sự bất đồng về từ phủ định giữa Đáp án và Luật/Tài liệu không
    choice_has_neg = any(word in choice_tokens for word in negation_words)
    doc_has_neg = any(word in doc_tokens for word in negation_words)

    # Nếu một bên có phủ định, một bên khẳng định -> Rất có thể mâu thuẫn
    if choice_has_neg != doc_has_neg:
        return True
    return False

def score_choice_universal(choice_text, doc_text, question_text, doc_metadata):
    """Chấm điểm đáp án đa lĩnh vực"""
    if not choice_text:
        return 0

    # 1. Điểm ngữ nghĩa tổng hợp (Truyền original_text để bắt từ viết tắt HOA)
    semantic_score = calculate_semantic_similarity(choice_text, doc_text, orig1=choice_text, orig2=doc_text)
    question_score = calculate_semantic_similarity(choice_text, question_text, orig1=choice_text, orig2=question_text)

    # 2. Khớp số liệu tuyệt đối (Vẫn cực kỳ quan trọng)
    numbers_choice = re.findall(r'\b\d+(?:[.,]\d+)?\b', preprocess_text(choice_text))
    numbers_doc = re.findall(r'\b\d+(?:[.,]\d+)?\b', preprocess_text(doc_text))

    number_score = 0
    if numbers_choice:
        matches = sum(1 for n in numbers_choice if n in numbers_doc)
        number_score = matches / len(numbers_choice)
    else:
        number_score = 0.5 # Default an toàn nếu đáp án là chữ

    # 3. Khớp cụm từ trong Tiêu đề (Metadata)
    metadata_score = 0
    choice_proc = preprocess_text(choice_text)
    for meta in doc_metadata:
        if choice_proc in preprocess_text(meta['title']):
            metadata_score += 0.5
            break

    # 4. Phạt nhẹ nếu dính bẫy phủ định
    if check_general_negation(choice_text, doc_text):
        semantic_score *= 0.6 # Phạt 40% điểm

    # Tổng hợp điểm
    final_score = (
        semantic_score * 0.50 +   # Từ vựng giống tài liệu là quan trọng nhất
        question_score * 0.10 +   # Liên kết với câu hỏi
        number_score * 0.30 +     # Khớp số liệu
        metadata_score * 0.10     # Khớp tiêu đề
    )
    return final_score

# ==========================================
# 4. MAIN LOOP
# ==========================================
def make_submission(test_file="de_thi.json", corpus_file="dataset.json",
                    output_file="submission.json", zip_file="submission.zip"):

    chunks, chunk_metadata = load_corpus_chunked(corpus_file)
    if not chunks:
        return

    print("Dang lap chi muc BM25 Da Linh Vuc...")
    tokenized_corpus = [preprocess_text(c).split() for c in chunks]
    bm25_model = BM25Okapi(tokenized_corpus)

    try:
        with open(test_file, 'r', encoding='utf-8') as f:
            test_data = json.load(f)
    except FileNotFoundError:
        print(f"Loi: Khong tim thay {test_file}")
        return

    submissions = []
    valid_choices = ["A", "B", "C", "D"]
    print(f"Bat dau lam {len(test_data)} cau trac nghiem...")

    for idx, item in enumerate(test_data):
        question_id = item.get("id")
        question_text = item.get("question", "")

        doc_text, top_metadata = find_best_document_bm25(
            question_text, chunks, chunk_metadata, bm25_model
        )

        choice_scores = {}
        for choice_key in valid_choices:
            choice_text = item.get(choice_key, "")
            choice_scores[choice_key] = score_choice_universal(
                choice_text, doc_text, question_text, top_metadata
            )

        best_choice = max(choice_scores, key=choice_scores.get) if max(choice_scores.values()) > 0 else "A"

        submissions.append({
            "id": question_id,
            "answer": best_choice
        })

        if idx % 50 == 0 or idx < 3:
            print(f"Cau {question_id} -> Dap an: {best_choice}")

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(submissions, f, ensure_ascii=False, indent=2)

    with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(output_file)
        code_file = os.path.basename(__file__) if '__file__' in globals() else 'main.ipynb'
        if os.path.exists(code_file):
            zipf.write(code_file)

    print(f"\nHoan thanh! Ket qua luu tai: {zip_file}")

    counts = Counter([s['answer'] for s in submissions])
    print("Phan bo dap an:")
    for c in valid_choices:
        print(f"  {c}: {counts.get(c, 0)} cau")

if __name__ == "__main__":
    make_submission()